<a href="https://colab.research.google.com/github/salinela/carbon-portfolio-project-v2/blob/main/notebooks/08_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# set up: desktop
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import sqlite3
import time
import seaborn as sns

# path set up:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

from src.config import DATA_RAW, DATA_PROCESSED
from src import eda

# database connection set up:
DB = ROOT/'data/carbon.db'
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")

In [3]:
# set up: Google Colab
import sys
import sqlite3
import time
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns

In [4]:

# mount drive (data artifacts live here — never in git)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# clone fresh, or pull if it already exists (so re-running the cell doesn't error)
import os
REPO = "/content/repo"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/salinela/carbon-portfolio-project-v2.git {REPO}
%cd {REPO}

Cloning into '/content/repo'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 195 (delta 115), reused 109 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 329.59 KiB | 7.16 MiB/s, done.
Resolving deltas: 100% (115/115), done.
/content/repo


In [6]:
# root on path — mirrors desktop's package-style imports
sys.path.insert(0, REPO)
print("root on path:", REPO)

root on path: /content/repo


In [7]:
# live-reload src edits after a git pull without restarting the runtime
from src import eda
from src import feature_engineering as fe
from src.config import DATA_RAW, DATA_PROCESSED

In [8]:
# DB copied to LOCAL disk (not the Drive FUSE mount) to avoid SQLite locking.
# Needed to read/query it, not just to rebuild — copy once per session.
DRIVE = "/content/drive/MyDrive/carbon_project_v2"
if not os.path.exists("/content/carbon.db"):
    !cp "{DRIVE}/carbon.db" /content/carbon.db

In [9]:
DB = "/content/carbon.db"
con = sqlite3.connect(DB, timeout=30)
con.execute("PRAGMA foreign_keys = ON;")

one-time usage

In [ ]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# one-time cleanup of the stale broken view
con.execute("DROP VIEW IF EXISTS v_company_emissions;")
con.commit()

# Phase 0: Data Assembly

In [17]:
# Build meta (summary dataframe/profile snapshot of all companies in the database) and meta_diag (summary dictionary of meta) for EDA:
meta, meta_diag = eda.build_meta(con)

# Build fy (yearly data: emissions and fundamentals) per company year:
fy,   fy_diag   = eda.build_firm_year(con)

meta_diag, fy_diag

({'n_companies': 8288,
  'by_universe': {'EU': 7988, 'ETS': 300},
  'eligible_n': 5883,
  'sector_nulls_master': 58,
  'sector_nulls_after_backfill': 58,
  'country_nulls': 0,
  'bvd_nulls': 0,
  'coverage_status_counts': {'mapped_loaded': 5883,
   'unmapped_exchange': 1773,
   'mapped_no_data': 523,
   'no_ticker': 109}},
 {'n_rows': 12487,
  'n_companies': 1306,
  'year_range': (2012, 2025),
  'by_source': {'trucost': 9214, 'ets_registry': 3273},
  'revenue_nulls': 573,
  'intensity_nulls': 573})

In [18]:
# summarising intial eligible companies:
fy_ids = set(fy["company_id"])
elig   = meta[meta["eligible"] == 1] # referring to no stock price series data available

mask   = elig.index.isin(fy_ids)
print("eligible:", len(elig))
print("eligible w/ emissions:", int(mask.sum()))
print(elig[mask]["universe"].value_counts().to_dict())

eligible: 5883
eligible w/ emissions: 1259
{'EU': 978, 'ETS': 281}


## Phase 1: Universe Characterization

### Section A: Compute carbon itensity tiers (source registry x 1-digit NACE x year)

In [24]:
# --- cohort flags on meta (enables optional carbon-blind comparison) ---
fy_ids = set(fy["company_id"])
meta["has_emissions_data"] = meta.index.isin(fy_ids).astype(int)
meta["carbon_sample"] = ((meta["eligible"] == 1) &
                         (meta["has_emissions_data"] == 1)).astype(int)

# cross-check against master's has_emissions flag:
print("has_emissions agree:",
      (meta["has_emissions"] == meta["has_emissions_data"]).mean())
print("carbon_sample n:", int(meta["carbon_sample"].sum()))

has_emissions agree: 0.9639237451737451
carbon_sample n: 1259


In [25]:
# --- nace1 for tiering: master, backfilled from orbis, first digit ---
nace_full = meta["nace_code"].fillna(meta["orbis_nace_code"])
nace1 = nace_full.astype("string").str.extract(r"(\d)")[0]   # extract nace1 code (first nace digit onky), index = company_id

# --- compute tiers using eda.compute_tiers on read; create carbon_tier at fy
fy, tier_diag = eda.compute_tiers(fy, nace1) # carbon_tier stored in fy
tier_diag

{'tier_counts': {'non_ets_high': 2947,
  'non_ets_low': 2911,
  'non_ets_medium': 2872,
  'ets_high': 1074,
  'ets_low': 1044,
  'ets_medium': 1015,
  <NA>: 573,
  'ets_untiered': 41,
  'non_ets_untiered': 10},
 'nace1_nulls': 0,
 'n_untiered': 51,
 'n_tiered': 11863}

In [29]:
# --- diagnose the difference between has_emission (master table in carbon.db) and has_emission_data (rejoined with emissions table):
d = meta[meta["has_emissions"] != meta["has_emissions_data"]]
print(len(d))
print(d.groupby(["has_emissions", "has_emissions_data"]).size())
print(d["universe"].value_counts().to_dict())

299
has_emissions  has_emissions_data
0              1                     299
dtype: int64
{'ETS': 299}


### Section B: Monthly Tier-based Portfolio Returns Helper

Main objective: for each month, take every firm currently sitting per tier and average their forward returns (equal-weighted basket)

tier_portfolio_returns produces one such series per tie; "do high-carbon baskets earn different returns than low-carbon ones"

The attach_tier_asof step is what tells each company-month which basket it was in at that date, using the 1-July lag so you're never using an emissions figure before it was public.

### Section B.1: Profiling Missingness in Tiered Month-End Features and Labels

We have two monthly price series dataframes a) **features** consisting of a suite of features derived from OHLVC data and c) **label** consisting of monthly forward returns; each monthly data comes with a carbon tier if emissions data are available

We first profile if there is any systematic NAs for specific month-end dates and provide a corresponding fix. Source of missing data:
- emission data --> untiered carbon status (likely for edge effects from too early/later dates but unlikely for in-between months)
- too little company in each bucket (low, medium, high)
- no monthly returns data
- no feature data

How do spot if there is missing data?
- see how many companies were in each bucket per year/month (if there is any thin cross-section)
- flag months with any missing data
- flag if the problem arise from no carbon data or returns data

Fix:
-

Desired outcome: No missingness per month (unless edge out-of-range months)

In [30]:
# panel (features and label):
features = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/features_month_end.parquet")
label    = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/label_fwd_return.parquet")

In [31]:
# pivot to wide:
panel = features.pivot_table(index=["company_id", "date"],
                             columns="signal_name", values="value")

# join features with label:
panel = panel.join(label.set_index(["company_id", "date"])["fwd_ret"])
print("panel:", panel.shape)

panel: (730161, 42)


In [49]:
# Attach carbon tiers to each monthly features panel:
panel_t = eda.attach_tier_asof(panel, fy)

# Attached tier NA rates across the whole data:
print("Tier coverage (monthly data with carbon tiers):", round(panel_t["carbon_tier"].notna().mean(), 3))

Tier coverage (monthly data with carbon tiers): 0.184


In [51]:
# Crude diagnostic reveal that low coverage arise from very little companies having carbon data:
p = panel.reset_index()

# Number of initial companies in the panel (monthly features/label data)
n_panel = p['company_id'].nunique()
n_fy = fy[fy['carbon_tier'].notna()]['company_id'].nunique()

# Companies with tier data compared to companies with features/label data:
print("Proportion of companies with carbon tier data compared to total companies with panel data: ",round(n_fy/n_panel,3))

Proportion of companies with carbon tier data compared to total companies with panel data:  0.215


In [ ]:
# Calculate averaged monthly returns per tier-month (equal-weighted, arithmetic); tier-dates with < min firms are NaN
tret = eda.tier_portfolio_returns(panel_t)


# Initial diagnostics (NA rates across the months per tier):
print(tret.isna().mean())

(155, 8)


carbon_tier,ets_high,ets_low,ets_medium,ets_untiered,non_ets_high,non_ets_low,non_ets_medium,non_ets_untiered
date,,,,,,,,
2026-01-30,NaN,NaN,NaN,NaN,0.001464,0.020621,-0.019596,NaN
2026-02-27,NaN,NaN,NaN,NaN,-0.057135,-0.056221,-0.082630,NaN
2026-03-31,NaN,NaN,NaN,NaN,0.044266,0.047350,0.072799,NaN
2026-04-30,NaN,NaN,NaN,NaN,0.043705,0.013508,0.048464,NaN
2026-05-29,NaN,NaN,NaN,NaN,-0.027886,-0.015544,-0.011606,NaN


From the initial missing rates:
- high

In [ ]:
# 1) firms actually CONTRIBUTING per tier per year (drives tret NaN directly)
chk = (panel_t.reset_index()[["company_id", "date", "carbon_tier", "fwd_ret"]]
       .dropna(subset=["carbon_tier", "fwd_ret"]))

# remove untiered:
chk = chk[~chk["carbon_tier"].astype(str).str.endswith("untiered")]

# see number of companies in each yearly tier:
chk["year"] = pd.to_datetime(chk["date"]).dt.year
chk.groupby(["year", "carbon_tier"])["company_id"].nunique().unstack("carbon_tier")

carbon_tier,ets_high,ets_low,ets_medium,non_ets_high,non_ets_low,non_ets_medium
year,,,,,,
2013,68.0,75.0,70.0,NaN,NaN,NaN
2014,81.0,87.0,92.0,144.0,148.0,140.0
2015,83.0,81.0,81.0,182.0,176.0,182.0
2016,87.0,86.0,83.0,183.0,177.0,189.0
2017,86.0,86.0,79.0,273.0,268.0,285.0
2018,91.0,86.0,83.0,304.0,307.0,316.0
2019,94.0,88.0,89.0,312.0,312.0,326.0
2020,95.0,91.0,93.0,327.0,334.0,353.0
2021,94.0,85.0,87.0,347.0,364.0,380.0


In [ ]:
tret_trim = tret.loc["2014-07-30":"2025-06-30"].drop(columns=['ets_untiered', 'non_ets_untiered'])
tret_trim.isna().mean()

,0
carbon_tier,
ets_high,0.015152
ets_low,0.030303
ets_medium,0.030303
non_ets_high,0.000000
non_ets_low,0.015152
non_ets_medium,0.015152


In [ ]:
tret_trim[tret_trim.isna().any(axis = 1)]

carbon_tier,ets_high,ets_low,ets_medium,non_ets_high,non_ets_low,non_ets_medium
date,,,,,,
2018-02-28,-0.035744,NaN,NaN,-0.031747,-0.014578,0.003955
2018-03-30,0.000761,NaN,NaN,-0.028747,0.063794,0.038382
2024-02-29,NaN,NaN,NaN,0.084667,NaN,NaN
2024-03-29,NaN,NaN,NaN,-0.023028,NaN,NaN


In [ ]:
probe = ["2018-02-28", "2018-03-30", "2024-02-29", "2024-03-29"]
px = panel_t.reset_index()
px["date"] = pd.to_datetime(px["date"]).astype(str)

# companies in specific dates
sub = px[px["date"].isin(probe) & px["carbon_tier"].notna()
         & ~px["carbon_tier"].astype(str).str.endswith("untiered")]

In [ ]:

# firms WITH a tier vs firms that also have a non-NaN fwd_ret, per tier-month
have_tier = sub.groupby(["date", "carbon_tier"])["company_id"].nunique()

have_ret  = (sub.dropna(subset=["fwd_ret"])
             .groupby(["date", "carbon_tier"])["company_id"].nunique())
pd.concat({"has_tier": have_tier, "has_ret": have_ret}, axis=1).fillna(0)

has_tier  has_ret
date       carbon_tier                      
2018-02-28 ets_high              84     14.0
           ets_low               85      1.0
           ets_medium            73      4.0
           non_ets_high         247     13.0
           non_ets_low          257      6.0
           non_ets_medium       243     17.0
2018-03-30 ets_high              63     14.0
           ets_low               45      1.0
           ets_medium            40      4.0
           non_ets_high         131     13.0
           non_ets_low          136      6.0
           non_ets_medium       149     17.0
2024-02-29 ets_high              77      2.0
           ets_low               76      0.0
           ets_medium            76      0.0
           non_ets_high         302      5.0
           non_ets_low          307      3.0
           non_ets_medium       305      1.0
2024-03-29 ets_high              34      2.0
           ets_low               25      0.0
           ets_medium            26      0.0
           non_ets_high          90      5.0
           non_ets_low           78      3.0
           non_ets_medium        95      1.0

A 1-month-forward return at end-Feb needs a price at end-Mar; end-March 2018 and end-March 2024 are Good Friday (2018-03-30, 2024-03-29 — markets shut). So:

At end-Feb, fwd_ret looks one month ahead to a closed Good Friday → NaN.
At end-Mar (the Good Friday itself), there's no price today → NaN.

That's why it's always a Feb/Mar pair, and only in years where Good Friday lands on the month-end. It's a forward-looking gap in the target, not a hole in the features. Which is why widening the window never helped — the NaN is baked into how the label was constructed.

In [ ]:
label_v2 = fe.forward_return_label(con, horizon=1, kind="log")

In [ ]:
label_v2.to_parquet(f"{DRIVE}/label_forward_return_v2.parquet")

j = label.merge(label_v2, on=["company_id","date"], suffixes=("_old","_new"))

print("existing values changed:",
      int((~np.isclose(j.fwd_ret_old, j.fwd_ret_new, atol=1e-9)).sum()), "of", len(j))
print("net new firm-month labels:", len(label_v2) - len(label))

existing values changed: 0 of 665683
net new firm-month labels: 73779


In [ ]:
# repoint the panel load at the fixed label
label = pd.read_parquet(f"{DRIVE}/label_forward_return_v2.parquet")
panel = features.pivot_table(index=["company_id","date"], columns="signal_name", values="value")
panel = panel.join(label.set_index(["company_id","date"])["fwd_ret"])

panel_t = eda.attach_tier_asof(panel, fy)
tret = eda.tier_portfolio_returns(panel_t).loc["2014-01-01":]
tret.isna().mean()

,0
carbon_tier,
ets_high,0.080537
ets_low,0.087248
ets_medium,0.087248
ets_untiered,0.926174
non_ets_high,0.040268
non_ets_low,0.046980
non_ets_medium,0.046980
non_ets_untiered,1.000000


In [ ]:
tret_trim = tret.loc["2014-07-30":"2025-06-30"].drop(columns=['ets_untiered', 'non_ets_untiered'])
tret_trim.isna().mean()

,0
carbon_tier,
ets_high,0.007576
ets_low,0.015152
ets_medium,0.015152
non_ets_high,0.000000
non_ets_low,0.007576
non_ets_medium,0.007576


In [ ]:
def label_coverage_scan(features_long, label_long, flag_frac=0.5):
    """Per month-end: firms with features vs firms with a non-NaN label. Flags
    months whose label/feature ratio is < flag_frac of the median ratio.
    Note: the final `horizon` month(s) legitimately have no label (no future)."""
    f = features_long.groupby("date")["company_id"].nunique().rename("n_features")
    l = label_long.groupby("date")["company_id"].nunique().rename("n_label")
    cov = pd.concat([f, l], axis=1).fillna(0)
    cov["ratio"] = cov["n_label"] / cov["n_features"].where(cov["n_features"] > 0)
    cov["flag"] = cov["ratio"] < flag_frac * cov["ratio"].median()
    return cov.sort_values("ratio")

In [ ]:
label_coverage = label_coverage_scan(features, label)

In [ ]:
label_coverage[label_coverage['flag'] == True]

,n_features,n_label,ratio,flag
date,,,,
2026-06-29,5820.0,0.0,0.000000,True
2024-03-29,1929.0,249.0,0.129082,True
2018-03-30,2444.0,728.0,0.297872,True
2013-03-29,1847.0,578.0,0.312940,True
2021-12-31,4670.0,2175.0,0.465739,True
2025-12-31,5289.0,2489.0,0.470599,True
2024-12-31,5172.0,2439.0,0.471578,True
2020-12-31,4248.0,2108.0,0.496234,True


In [ ]:
features = pd.read_parquet(f"{DRIVE}/features_month_end.parquet")
label    = pd.read_parquet(f"{DRIVE}/label_forward_return_v2.parquet")

# canonical monthly key: snap each obs to its calendar month-end
for d in (features, label):
    d["date"] = pd.to_datetime(d["date"]) + pd.offsets.MonthEnd(0)

panel = features.pivot_table(index=["company_id","date"], columns="signal_name", values="value")
panel = panel.join(label.set_index(["company_id","date"])["fwd_ret"])

panel_t = eda.attach_tier_asof(panel, fy)
tret = eda.tier_portfolio_returns(panel_t).loc["2014-01-01":]
print(tret.isna().mean())

cov = eda.label_coverage_scan(features, label)
print(cov[cov.flag])

carbon_tier
ets_high            0.073826
ets_low             0.073826
ets_medium          0.073826
ets_untiered        0.919463
non_ets_high        0.040268
non_ets_low         0.040268
non_ets_medium      0.040268
non_ets_untiered    1.000000
dtype: float64
            n_features  n_label  ratio  flag
date                                        
2026-06-30        5820      0.0    0.0  True


In [ ]:
tret_trim = tret.loc["2014-07-30":"2025-06-30"].drop(columns=['ets_untiered', 'non_ets_untiered'])
print(tret_trim.isna().mean())

carbon_tier
ets_high          0.0
ets_low           0.0
ets_medium        0.0
non_ets_high      0.0
non_ets_low       0.0
non_ets_medium    0.0
dtype: float64


### Section B.2: Month-end data per tier

In [ ]:
# 1) firms actually CONTRIBUTING per tier per year (drives tret NaN directly)
chk = (panel_t.reset_index()[["company_id", "date", "carbon_tier", "fwd_ret"]]
       .dropna(subset=["carbon_tier", "fwd_ret"]))
chk = chk[~chk["carbon_tier"].astype(str).str.endswith("untiered")]
chk["year"] = pd.to_datetime(chk["date"]).dt.year
chk.groupby(["year", "carbon_tier"])["company_id"].nunique().unstack("carbon_tier")

carbon_tier,ets_high,ets_low,ets_medium,ets_untiered,non_ets_high,non_ets_low,non_ets_medium,non_ets_untiered
date,,,,,,,,
2014-01-31,0.058386,0.039097,0.056950,NaN,NaN,NaN,NaN,NaN
2014-02-28,-0.006674,0.000626,0.012330,NaN,NaN,NaN,NaN,NaN
2014-03-31,0.053258,0.003538,-0.010680,NaN,NaN,NaN,NaN,NaN
2014-04-30,0.019536,0.015005,0.014068,NaN,NaN,NaN,NaN,NaN
2014-05-31,-0.018652,0.000140,-0.001501,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2026-01-31,NaN,NaN,NaN,NaN,0.001464,0.020621,-0.019596,NaN
2026-02-28,NaN,NaN,NaN,NaN,-0.057135,-0.056221,-0.082630,NaN
2026-03-31,NaN,NaN,NaN,NaN,0.044266,0.047350,0.072799,NaN


In [ ]:
# 2) is the bottleneck upstream (emissions/revenue) or the price panel?
fyt = fy[fy["carbon_tier"].notna()
         & ~fy["carbon_tier"].astype(str).str.endswith("untiered")]
fyt.groupby(["year", "source"]).size().unstack("source")